# Data Transformation and Feature Engineering
**Unit 3 · Lesson 5 | PV Time Series**

Load Lesson 2 supervised partitions → explicitly select input roles → ordinal encode weather → standardize candidate inputs → save CSVs and fitted preprocessing.

Keep three versions per split: **raw** (Lesson 2 CSV, unchanged), **encoded** (weather encoded, numerical measurements in original units), and **standardized** (encoded + scaled). Both targets and all time metadata remain unchanged in every version. No repeated target construction, inverse target scaling, EDA or automatic cleaning is needed.

## 1. Load the prepared partitions
Run locally from `chapter_03/`, or edit `PROJECT_ROOT`. Shared data lives in `data/processed/pv_time_series/`, outside chapter folders.

**Colab:** runtime storage is temporary. Keep the original Lesson 2 CSVs and generated outputs on Drive or download them before ending the session. Saving a notebook does not save runtime CSVs. Use a mounted Drive project root or the optional `gdown` example.

Run the imports, then choose **one** loading-location option below. Local setup is active by default; uncomment the alternative you need. Then run the shared CSV-reading cell.

In [18]:
# %pip install "scikit-learn>=1.2" pandas joblib
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import joblib
import sklearn
import gdown

### Option A — Local project

In [19]:
# PROJECT_ROOT = Path("..").resolve()
# DATA_DIR = PROJECT_ROOT / "data/processed/pv_time_series"

### Option B — Google Colab with Google Drive
Uncomment this cell to use persistent Drive storage.

In [20]:
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = Path("/content/drive/MyDrive/pv_course")
# DATA_DIR = PROJECT_ROOT / "data/processed/pv_time_series"

### Option C — Download prepared CSVs using gdown
Replace the file IDs and uncomment. These downloads use temporary Colab storage; preserve the outputs before ending the session.

In [21]:
DATA_DIR = Path("/content/data/processed/pv_time_series")
DATA_DIR.mkdir(parents=True, exist_ok=True)
# gdown.download(id="TRAIN_FILE_ID", output=str(DATA_DIR / "pv_time_series_train.csv"), quiet=False)
# gdown.download(id="VALIDATION_FILE_ID", output=str(DATA_DIR / "pv_time_series_validation.csv"), quiet=False)
# gdown.download(id="TEST_FILE_ID", output=str(DATA_DIR / "pv_time_series_test.csv"), quiet=False)
gdown.download(id="1dNy81OHu4fIECJshylPL-b36k8U4wreF", output=str(DATA_DIR / "pv_time_series_train.csv"), quiet=False)
gdown.download(id="1CY0ny1jPb1JSqfaZZ583GeZtuRp56Md_", output=str(DATA_DIR / "pv_time_series_validation.csv"), quiet=False)
gdown.download(id="1IB0qv5K87J6MGmc8Q7SG-Q0lC745In1r", output=str(DATA_DIR / "pv_time_series_test.csv"), quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1dNy81OHu4fIECJshylPL-b36k8U4wreF
To: /content/data/processed/pv_time_series/pv_time_series_train.csv
100%|██████████| 718k/718k [00:00<00:00, 52.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1CY0ny1jPb1JSqfaZZ583GeZtuRp56Md_
To: /content/data/processed/pv_time_series/pv_time_series_validation.csv
100%|██████████| 160k/160k [00:00<00:00, 73.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1IB0qv5K87J6MGmc8Q7SG-Q0lC745In1r
To: /content/data/processed/pv_time_series/pv_time_series_test.csv
100%|██████████| 153k/153k [00:00<00:00, 19.2MB/s]


'/content/data/processed/pv_time_series/pv_time_series_test.csv'

### Read the three CSVs
Run after choosing a data location.

In [22]:
train_data = pd.read_csv(DATA_DIR / "pv_time_series_train.csv")
validation_data = pd.read_csv(DATA_DIR / "pv_time_series_validation.csv")
test_data = pd.read_csv(DATA_DIR / "pv_time_series_test.csv")

print("Train:", train_data.shape)
print("Validation:", validation_data.shape)
print("Test:", test_data.shape)
display(train_data.head())

Train: (4955, 25)
Validation: (1061, 25)
Test: (1061, 25)


,day,cosine_of_day,hour,cosine_of_hour,air_temperature,dewpoint,wind_speed,air_pressure,weather_condition,solar_radiation,...,pv_3_power,pv_4_cosine_value_of_the_solar_incidence_angle,pv_4_power,pv_5_cosine_value_of_the_solar_incidence_angle,pv_5_power,total_pv_power,time_index,target_time_index,target_power_t_plus_1,target_class_t_plus_1
0,1,0.999852,1,1.0000,2,81,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0,0,1,0.0,low
1,1,0.999852,2,0.9659,-1,86,0,900.2,10,0.0,...,0,0.0,0,0.0,0,0,1,2,0.0,low
2,1,0.999852,3,0.8660,0,87,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0,2,3,0.0,low
3,1,0.999852,4,0.7071,-1,86,2,900.2,10,0.0,...,0,0.0,0,0.0,0,0,3,4,0.0,low
4,1,0.999852,5,0.5000,-1,86,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0,4,5,0.0,low


### Explicit column roles
Only `weather_condition` is a nominal input to ordinal-encode. Time keys and both labels must remain outside feature transformations. All original source columns are preserved; 19 columns are candidate inputs and two original time labels remain metadata.

In [23]:
TARGET_COLUMNS = ["target_power_t_plus_1", "target_class_t_plus_1"]
TIME_COLUMNS = ["day", "hour", "time_index", "target_time_index"]
CATEGORICAL_COLUMNS = ["weather_condition"]
CONTINUOUS_COLUMNS = [
    "cosine_of_day", "cosine_of_hour", "air_temperature", "dewpoint",
    "wind_speed", "air_pressure", "solar_radiation",
    "pv_1_cosine_value_of_the_solar_incidence_angle", "pv_1_power",
    "pv_2_cosine_value_of_the_solar_incidence_angle", "pv_2_power",
    "pv_3_cosine_value_of_the_solar_incidence_angle", "pv_3_power",
    "pv_4_cosine_value_of_the_solar_incidence_angle", "pv_4_power",
    "pv_5_cosine_value_of_the_solar_incidence_angle", "pv_5_power",
    "total_pv_power",
]
FEATURE_COLUMNS = CONTINUOUS_COLUMNS + CATEGORICAL_COLUMNS
SCALE_COLUMNS = FEATURE_COLUMNS.copy()
expected = set(FEATURE_COLUMNS + TIME_COLUMNS + TARGET_COLUMNS)
for name, data in [("train", train_data), ("validation", validation_data), ("test", test_data)]:
    if set(data.columns) != expected:
        raise ValueError(f"{name}: rerun updated Lesson 2; missing={expected-set(data.columns)}, unexpected={set(data.columns)-expected}")
print("Continuous inputs:", len(CONTINUOUS_COLUMNS))
print("Categorical inputs:", CATEGORICAL_COLUMNS)
print("Candidate model input count:", len(FEATURE_COLUMNS))

Continuous inputs: 18
Categorical inputs: ['weather_condition']
Candidate model input count: 19


## 2. Ordinal encode the weather code
Keep weather as **one input column**, rather than expanding it into multiple indicator columns. `OrdinalEncoder` learns the category vocabulary from train only and assigns consecutive numeric codes. Apply the same mapping to validation/test.

For this training file, source codes `[1, 2, 3, 4, 5, 6, 7, 9, 10]` map to `[0, 1, 2, 3, 4, 5, 6, 7, 8]`. The table below shows the actual learned mapping. An unseen category receives `-1` and is reported; it is not added to the training vocabulary.

**Trade-off:** this compact representation is a project choice, not evidence that weather conditions have a natural order. A neural network consuming these numbers as a scalar sees numeric order and distances that may not reflect category meaning. Standardization does not remove that assumption. Evaluate its effect during modeling; ordinal encoding is not universally better than one-hot encoding.

We retain the existing cosine features and do not add extra hour sine/cosine columns without evaluating their need. No log transforms, interactions or automatic feature deletion are added just to cover slide topics.

In [24]:
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(train_data[["weather_condition"]])
display(pd.DataFrame({
    "original_weather_code": encoder.categories_[0],
    "encoded_weather_code": range(len(encoder.categories_[0])),
}))

def encode_weather(data):
    unknown = ~data["weather_condition"].isin(encoder.categories_[0])
    print("Rows with unknown weather codes:", int(unknown.sum()))
    result = data.copy()
    result["weather_condition"] = encoder.transform(data[["weather_condition"]]).ravel()
    return result

train_encoded = encode_weather(train_data)
validation_encoded = encode_weather(validation_data)
test_encoded = encode_weather(test_data)
display(train_encoded[["weather_condition"]].head())

,original_weather_code,encoded_weather_code
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4
5,6,5
6,7,6
7,9,7
8,10,8


Rows with unknown weather codes: 0
Rows with unknown weather codes: 0
Rows with unknown weather codes: 0


,weather_condition
0,8.0
1,8.0
2,8.0
3,8.0
4,8.0


## 3. Standardize only explicitly selected inputs
Use `StandardScaler` fitted on the encoded training input columns only. Apply that same scaler to validation/test.

- Scale all 18 continuous candidate inputs: meteorology (including `dewpoint`), existing cosines, five panel outputs and current total output.
- Scale the single ordinal weather input as a scalar too. This is a modeling convention, not a natural weather ordering. Its unknown value `-1` is scaled by the same training statistics.
- Do **not** scale or encode either target. Regression labels stay in source units; classification labels stay low/medium/high.
- Keep `day`, `hour`, `time_index`, `target_time_index` unchanged as metadata, not default inputs. Existing cosines remain inputs; standardizing them is allowed even though they are already bounded.

Standardization is not min–max normalization and does not remove ordinal distance assumptions. Candidate inputs are not a final feature selection: verify timing and the meaning of `dewpoint` before modeling. Current `total_pv_power` is an input at t, unlike the future regression label.

An allowlist is safer than “all columns except day/hour”: adding targets must never silently include them in X or scaler fitting.

In [25]:
scaler = StandardScaler()
scaler.fit(train_encoded[SCALE_COLUMNS])

def standardize_features(data):
    result = data.copy()
    result[SCALE_COLUMNS] = scaler.transform(data[SCALE_COLUMNS])
    return result

train_standardized = standardize_features(train_encoded)
validation_standardized = standardize_features(validation_encoded)
test_standardized = standardize_features(test_encoded)
display(train_standardized.head())

,day,cosine_of_day,hour,cosine_of_hour,air_temperature,dewpoint,wind_speed,air_pressure,weather_condition,solar_radiation,...,pv_3_power,pv_4_cosine_value_of_the_solar_incidence_angle,pv_4_power,pv_5_cosine_value_of_the_solar_incidence_angle,pv_5_power,total_pv_power,time_index,target_time_index,target_power_t_plus_1,target_class_t_plus_1
0,1,1.523977,1,1.413803,-1.000479,1.166969,-0.905450,0.605465,0.892086,-0.713688,...,-0.679291,-0.767549,-0.698776,-0.789662,-0.712705,-0.70617,0,1,0.0,low
1,1,1.523977,2,1.365573,-1.300574,1.398466,-1.604610,0.605465,0.892086,-0.713688,...,-0.679291,-0.767549,-0.698776,-0.789662,-0.712705,-0.70617,1,2,0.0,low
2,1,1.523977,3,1.224278,-1.200542,1.444765,-0.905450,0.605465,0.892086,-0.713688,...,-0.679291,-0.767549,-0.698776,-0.789662,-0.712705,-0.70617,2,3,0.0,low
3,1,1.523977,4,0.999536,-1.300574,1.398466,-1.371557,0.605465,0.892086,-0.713688,...,-0.679291,-0.767549,-0.698776,-0.789662,-0.712705,-0.70617,3,4,0.0,low
4,1,1.523977,5,0.706621,-1.300574,1.398466,-0.905450,0.605465,0.892086,-0.713688,...,-0.679291,-0.767549,-0.698776,-0.789662,-0.712705,-0.70617,4,5,0.0,low


## 4. Save the CSVs
Save each table directly beside the prepared CSVs with a new name. Choose the encoded or standardized version later according to the model. Rerunning these cells refreshes only these output files, not the original split CSVs.

### Encoded observations — original numerical units

In [26]:
train_encoded.to_csv(DATA_DIR / "pv_time_series_train_encoded.csv", index=False)
validation_encoded.to_csv(DATA_DIR / "pv_time_series_validation_encoded.csv", index=False)
test_encoded.to_csv(DATA_DIR / "pv_time_series_test_encoded.csv", index=False)

### Standardized observations

In [27]:
train_standardized.to_csv(DATA_DIR / "pv_time_series_train_standardized.csv", index=False)
validation_standardized.to_csv(DATA_DIR / "pv_time_series_validation_standardized.csv", index=False)
test_standardized.to_csv(DATA_DIR / "pv_time_series_test_standardized.csv", index=False)

## 5. Save fitted preprocessing for reuse
Save the training-fitted encoder, scaler, exact column order and software versions together. This avoids refitting on inference data or guessing which columns were scaled. Use a compatible environment and load **only trusted** joblib files: deserialization can execute code.

Class thresholds are defined in Lesson 2 and reused unchanged here; they are not part of feature scaling.

In [28]:
preprocessing = {
    "encoder": encoder,
    "scaler": scaler,
    "feature_columns": FEATURE_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "scale_columns": SCALE_COLUMNS,
    "time_columns": TIME_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "versions": {"scikit_learn": sklearn.__version__, "pandas": pd.__version__, "joblib": joblib.__version__},
}
joblib.dump(preprocessing, DATA_DIR / "pv_time_series_preprocessing.joblib")

['/content/data/processed/pv_time_series/pv_time_series_preprocessing.joblib']

## 6. Handoff to modeling
- Raw: `pv_time_series_<split>.csv` from Lesson 2, unchanged.
- Encoded: `pv_time_series_<split>_encoded.csv` (ordinal weather; other measurements in source units).
- Encoded + scaled: `pv_time_series_<split>_standardized.csv`.
- Use `FEATURE_COLUMNS` for X, and choose **one** `TARGET_COLUMNS` entry for y. Neither target is an input to either task. Do not standardize standardized CSVs again.
- Classification labels are already available: no inverse transform, new shift or recalculation from current output is needed. A classifier needing integer labels can use a fixed output-label mapping at modeling time; that is separate from weather encoding.
- Regression predictions are in source units when fitted against the raw target; there is no target scaler in this lesson.
- For temporal cross-validation, refit encoding, scaling and data-derived class thresholds within each training fold, starting from untransformed inputs. Do not reuse this global training preprocessing across folds.
- Preserve all CSVs, preparation metadata and the joblib artifact on persistent storage in Colab. Confirm source units and input availability before modeling.